# Quality audit and limitations

<!-- Drafted with the help of an AI assistant (Claude, Anthropic), disclosed per course rules. -->

How correct is the output? Three checks, from the cheapest to the most expensive: integrity of the tables, agreement of the tree with the source, and a reading of random papers.

## 1. Integrity of the tables

`python -m src.qa.check_tables` verifies, on the whole output, the column types of the statement, that `element_id` is unique, that every `parent_id` exists, that each version has exactly one root from which every node is reachable, and that a branch has a single parent. The output of the 100-paper run passes all checks (43,353 elements, 55,885 hierarchy rows, 3,800 bibliography rows, 0 problems).

## 2. Agreement with the source

The table of the previous chapter ("Does the tree keep what the source has?") is the second check: in the 121 parsed versions the number of headings and of floats is the same in the source and in the tree, list items agree in 119 and displayed formulas in 110.

## 3. Reading 30 random papers

`python -m src.qa.audit_sample draw ... -n 30 --seed 2026` draws 30 publications at random from the parsed ones and writes, for each, the tree rebuilt from the tables next to the source, and a blank score sheet. The 30 publications of the smoke crawl were read this way with the help of an AI assistant (the rendered outline, sentences, equations, figures and bibliography entries against the source), and the same sentences were scanned automatically for leftovers of LaTeX (a sentence that starts with a command, unbalanced braces, a macro parameter). **The score sheet is meant to be filled by a team member; the table below reports what the reading and the scan found, not a filled sheet.**

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path("../data/m2")
findings = pd.read_csv(DATA / "audit_findings.csv")
flags = pd.read_csv(DATA / "sentence_flags.csv")

In [ ]:
findings.set_index('category')

In [ ]:
flags.set_index('scope')

## Known limitations

- **Author macros are not expanded.** A sentence such as `\toolname achieves ...` keeps the macro name; `\newcommand` definitions are dropped, not applied.
- **Template front matter.** The commands of the common journal templates are dropped, but a template the parser has not seen can leave its author block or its notes among the sentences of the first section (15 of the 30 sampled publications showed this on the first reading, 5 after the fixes made while reading).
- **References are heuristics.** `citation_key`, `year`, `doi` and `url` are reliable; `authors`, `title`, `journal` and `entry_type` depend on the style of a hand-written `\bibitem`. A version whose references exist only in a `.bbl` that was dropped by the earlier crawl rule has an empty reference list (about one version in ten in the smoke crawl); re-crawling those versions with the `.bbl` fallback fills it.
- **Equal elements under one parent are stored once**, so a sentence repeated inside the same section appears once in `hierarchy`, and the order of siblings is the order of the rows, not a column.
- **Not followed:** `\subfile`, `\import` and `\InputIfFileExists`.
- **The numbers come from 100 publications.** They show behaviour, not the corpus.